In [2]:
import numpy as np
import torch
import clip
from transformers import AutoImageProcessor, AutoModel
from qdrant_client import QdrantClient, models
import pandas as pd
from PIL import Image
import os
import time

os.getpid()

/home/inna/.cache/pypoetry/virtualenvs/sneakersearch-z_mDpiHd-py3.12/lib/python3.12/site-packages/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
/home/inna/.cache/pypoetry/virtualenvs/sneakersearch-z_mDpiHd-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4507

In [3]:
#поднимаем ВБД с маппингом в корень прокта
# в корне проекта выполнить 

#docker run -p 6333:6333 -p 6334:6334 -v "$(pwd)/qdrant_storage:/qdrant/storage" qdrant/qdrant

# инициализация клиента ВБД
client = QdrantClient("http://localhost:6333")

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
# client.recreate_collection(
#         collection_name="sneakers",
#         vectors_config=models.VectorParams(size=512, distance=models.Distance.COSINE),
#     )

/tmp/ipykernel_15972/1366355860.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [12]:
prefix_path = "/home/inna/Рабочий стол/SneakerSearch/data/"

lamoda_data = pd.read_csv(prefix_path+"lamoda_data.csv", sep=";")
lamoda_data

,brand,model,category,color,description,lamoda_photo,title_photo,path_to_lamoda_photo
0,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
1,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
2,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
3,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
4,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
...,...,...,...,...,...,...,...,...
1342,Reebok,Reebok Кроссовки ULTRA LO,Низкие кроссовки,зеленый,Кроссовки выполнены из текстиля и натурально к...,https://a.lmcdn.ru/product/R/T/RTLAFA084101_31...,lamoda_photos/Reebok_Reebok_Кроссовки_ULTRA_LO...,//a.lmcdn.ru/product/R/T/RTLAFA084101_31852372...
1343,Reebok,Reebok Кроссовки ULTRA LO,Низкие кроссовки,зеленый,Кроссовки выполнены из текстиля и натурально к...,https://a.lmcdn.ru/product/R/T/RTLAFA084101_31...,lamoda_photos/Reebok_Reebok_Кроссовки_ULTRA_LO...,//a.lmcdn.ru/product/R/T/RTLAFA084101_31852372...
1344,Reebok,Reebok Кроссовки ULTRA LO,Низкие кроссовки,зеленый,Кроссовки выполнены из текстиля и натурально к...,https://a.lmcdn.ru/product/R/T/RTLAFA084101_31...,lamoda_photos/Reebok_Reebok_Кроссовки_ULTRA_LO...,//a.lmcdn.ru/product/R/T/RTLAFA084101_31852372...
1345,Reebok,Reebok Кроссовки ULTRA LO,Низкие кроссовки,зеленый,Кроссовки выполнены из текстиля и натурально к...,https://a.lmcdn.ru/product/R/T/RTLAFA084101_31...,lamoda_photos/Reebok_Reebok_Кроссовки_ULTRA_LO...,//a.lmcdn.ru/product/R/T/RTLAFA084101_31852372...


In [13]:
# оставляем только записи с титульными фото
lamoda_data = lamoda_data.drop_duplicates(subset="path_to_lamoda_photo")
print(len(lamoda_data))

225


In [14]:
lamoda_data = lamoda_data.drop_duplicates(subset="title_photo")
print(len(lamoda_data))

214


## CLIP emb

In [39]:
# инизацлизация модели для ембеддингов
model_name = "ViT-B/32" #338M params
model, preprocess = clip.load(model_name, device=device) 

In [50]:
img_path = prefix_path+lamoda_data.iloc[1]["title_photo"]
image = Image.open(img_path).convert("RGB")
preproc_image = preprocess(image).unsqueeze(0).to(device)
with torch.no_grad():
    image_emb = model.encode_image(preproc_image)
image_emb.shape

torch.Size([1, 512])

In [54]:
def get_image_embedding(img_path):
    image = Image.open(img_path).convert('RGB')

    preproc_image = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        image_emb = model.encode_image(preproc_image)
        
    # Нормализуем эмбеддинг (это важно для поиска в Qdrant)
    image_emb /= image_emb.norm(dim=-1, keepdim=True)
    
    return image_emb.cpu().numpy().flatten()

In [57]:
def create_db(
        client: QdrantClient,
        collection_name: str, 
        emb_dim: int,
        data):
    
    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(size=emb_dim, distance=models.Distance.COSINE),
        )
        unprocessable = 0

        for idx, row in data.iterrows():
                img_path = prefix_path+lamoda_data.loc[idx]["title_photo"]
                try:
                    image_emb = get_image_embedding(img_path)

                    point = models.PointStruct(
                        id=idx, 
                        vector=image_emb.tolist(), 
                        payload={
                            "brand": row["brand"],
                            "model": row["model"],
                            "color": row["color"],
                            "path_to_photo": os.path.join(prefix_path, row["title_photo"]) # cохраняем путь для отображения!
                        }
                    )

                    client.upsert(
                        collection_name=collection_name, 
                        points=[point]
                        )
                except:
                    unprocessable+=1 
                    continue
        print("unprocessable: ", unprocessable)
    else:
        print("Коллекция уже существует")

In [58]:
emb_dim = 512
collection_name = "Sneakers_CLIP"

create_db(
        client,
        collection_name, 
        emb_dim,
        lamoda_data)

unprocessable:  0


## Metrics CLIP

In [19]:
user_data = pd.read_csv(prefix_path+"user_data.csv", sep=";")
user_data = user_data.drop_duplicates()
user_data = user_data.drop_duplicates(subset="path_to_user_photo")
user_data = user_data.sample(frac=1, random_state=42).reset_index(drop=True) #shuffle data
print(len(user_data))

276


In [20]:
user_data = pd.read_csv(prefix_path+"user_data.csv", sep=";")
user_data = user_data.drop_duplicates()
user_data = user_data.drop_duplicates(subset="path_to_user_photo")
user_data = user_data.sample(frac=1, random_state=42).reset_index(drop=True) #shuffle data
print(len(user_data))

276


In [21]:
user_data

,brand,model,category,color,description,title_photo,path_to_title_photo,user_photo,path_to_user_photo
0,Baasploa,Baasploa Кроссовки,Низкие кроссовки,63085,,//a.lmcdn.ru/product/M/P/MP002XW0OZFJ_22116294...,lamoda_photos/Baasploa_Baasploa_Кроссовки__0.jpg,https://a.lmcdn.ru/photoreview/?key=80011863-a...,user_photos/Baasploa_Baasploa_Кроссовки__2.jpg
1,Mango,Mango Кроссовки SOFI,Низкие кроссовки,26244,NaN,//a.lmcdn.ru/product/R/T/RTLAEX902501_31509501...,lamoda_photos/Mango_Mango_Кроссовки_SOFI_0.jpg,https://a.lmcdn.ru/photoreview/?key=156d67a6-e...,user_photos/Mango_Mango_Кроссовки_SOFI_1.jpg
2,PUMA,PUMA Кроссовки Fade Nitro Ripstop,Низкие кроссовки,77715,"Текстильный верх усилен плетением рипстоп, кот...",//a.lmcdn.ru/product/R/T/RTLAFC584301_32686497...,lamoda_photos/PUMA_PUMA_Кроссовки_Fade_Nitro_R...,https://a.lmcdn.ru/photoreview/?key=5eb423cc-b...,user_photos/PUMA_PUMA_Кроссовки_Fade_Nitro_Rip...
3,Vans,Vans Кеды Authentic,Низкие кеды,26213,Кеды выполнены из текстиля. Детали: шнуровка н...,//a.lmcdn.ru/product/R/T/RTLAFE050301_33176676...,lamoda_photos/Vans_Vans_Кеды_Authentic_0.jpg,https://a.lmcdn.ru/photoreview/?key=f1b3c74a-8...,user_photos/Vans_Vans_Кеды_Authentic_1.jpg
4,Saucony,Saucony Кроссовки AURA TR,Низкие кроссовки,48066,NaN,//a.lmcdn.ru/product/R/T/RTLAET683701_30852120...,lamoda_photos/Saucony_Saucony_Кроссовки_AURA_T...,https://a.lmcdn.ru/photoreview/?key=5fd13683-e...,user_photos/Saucony_Saucony_Кроссовки_AURA_TR_...
...,...,...,...,...,...,...,...,...,...
271,adidas,adidas Кроссовки Ligra 8 M,Низкие кроссовки,86747,Кроссовки выполнены из искусственной кожи и те...,//a.lmcdn.ru/product/R/T/RTLAEU498901_31149950...,lamoda_photos/adidas_adidas_Кроссовки_Ligra_8_...,https://a.lmcdn.ru/photoreview/?key=6a7752cc-8...,user_photos/adidas_adidas_Кроссовки_Ligra_8_M_...
272,Columbia,Columbia Кроссовки CRESTWOOD™,Низкие кроссовки,10099,Кроссовки с комбинированным верхом из натураль...,//a.lmcdn.ru/product/M/P/MP002XW02YFS_12636285...,lamoda_photos/Columbia_Columbia_Кроссовки_CRES...,https://a.lmcdn.ru/photoreview/?key=ff8ec54a-6...,user_photos/Columbia_Columbia_Кроссовки_CRESTW...
273,Ecco,Ecco Кеды SOFT 7 W,Низкие кеды,547,NaN,//a.lmcdn.ru/product/M/P/MP002XW0S0Z2_11057679...,lamoda_photos/Ecco_Ecco_Кеды_SOFT_7_W_0.jpg,https://a.lmcdn.ru/photoreview/?key=decae862-b...,user_photos/Ecco_Ecco_Кеды_SOFT_7_W_0.jpg
274,Kappa,Kappa Кроссовки SELECTO MD,Кроссовки,93057,"Кроссовки выполнены из синтетической кожи, доп...",//a.lmcdn.ru/product/M/P/MP002XW0OTPY_22083371...,lamoda_photos/Kappa_Kappa_Кроссовки_SELECTO_MD...,https://a.lmcdn.ru/photoreview/?key=70c0ac26-b...,user_photos/Kappa_Kappa_Кроссовки_SELECTO_MD_2...


In [22]:
test_data = user_data[-100:]
image_reference = dict(zip(test_data["path_to_user_photo"], test_data["model"]))

In [23]:
def count_metrics(collection_name):

    recall_5 = 0
    recall_10 = 0
    accuracy = 0
    query_time = []
    N = 0
    unprocessable = []

    for image_path, reference_model in image_reference.items():
        try:
            start = time.perf_counter()
            query = get_image_embedding(prefix_path+image_path)
            candidates = client.query_points(
                                        collection_name=collection_name,
                                        query=query, 
                                        limit=10,           
                                        with_payload=True   # Возвращаем бренд, модель и путь из CSV
                                    ).points
            end = time.perf_counter()
        
            results = [candidate.payload.get("model") for candidate in candidates]
            if reference_model in results:
                recall_10 += 1
            if reference_model in results[:5]:
                recall_5 += 1
            if reference_model == results[0]:
                accuracy += 1
            query_time.append(end-start)
            N +=1

        except:
            unprocessable.append(image_path)
            continue

    print("Unprocessable: ", len(unprocessable))

    return recall_10 *100 / N, recall_5 * 100 / N, accuracy * 100 / N, np.mean(query_time)

In [59]:
recall_10_clip, recall_5_clip, accuracy_clip, avg_query_time_clip = count_metrics(collection_name)

Unprocessable:  0


In [60]:
df_data = {
    "Accuracy" : [accuracy_clip],
    "Recall@5": [recall_5_clip], 
    "Recall@10": [recall_10_clip],
    "Avg_query_time": [avg_query_time_clip]
}
  
table = pd.DataFrame(data=df_data)
table

,Accuracy,Recall@5,Recall@10,Avg_query_time
0,25.0,45.0,54.0,0.263479


## DINO emb

In [62]:
model_name = 'facebook/dinov2-large' #300M params


processor = AutoImageProcessor.from_pretrained(model_name)
model_dino = AutoModel.from_pretrained(model_name, device_map="auto")

Loading weights: 100%|██████████| 439/439 [00:00<00:00, 443.19it/s]


In [63]:
img_path = "/home/inna/Рабочий стол/SneakerSearch/scripts/lamoda_photos/Under_Armour_Under_Armour_Кроссовки_UA_W_Charged_P_0.jpg"
image = Image.open(img_path).convert("RGB")


inputs = processor(images=image, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = model_dino(**inputs)
    image_emb = outputs.last_hidden_state[:, 0, :] # CLS-token

image_emb.shape

torch.Size([1, 1024])

In [64]:
def get_image_embedding(img_path):
    image = Image.open(img_path).convert('RGB')
    
    preproc_image = processor(images=image, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = model_dino(**preproc_image)
        image_emb = outputs.last_hidden_state[:, 0, :] # CLS-token
        # Нормализуем эмбеддинг (это важно для поиска в Qdrant)
        image_emb /= image_emb.norm(dim=-1, keepdim=True)
    
    return image_emb.cpu().numpy().flatten()

In [65]:
emb_dim = 1024
collection_name = "Sneakers_DINO"

create_db(
        client,
        collection_name, 
        emb_dim,
        lamoda_data)

unprocessable:  0


In [66]:
recall_10_dino, recall_5_dino, accuracy_dino, avg_query_time_dino = count_metrics(collection_name)

Unprocessable:  0


In [67]:
df_data = {
    "Accuracy" : [accuracy_clip, accuracy_dino],
    "Recall@5": [recall_5_clip, recall_5_dino], 
    "Recall@10": [recall_10_clip, recall_10_dino],
    "Avg_query_time": [avg_query_time_clip, avg_query_time_dino]
}
  
table = pd.DataFrame(data=df_data)
table

,Accuracy,Recall@5,Recall@10,Avg_query_time
0,25.0,45.0,54.0,0.263479
1,3.0,14.0,26.0,0.195054


## Mультивекторность Qdrant

**ЧТО:** 

В Qdrant реализованы два основных подхода к работе с несколькими векторами:
1. Именованные векторы (Named Vectors). Этот метод используется, когда один объект нужно описать разными характеристиками или модальностями. Если объединить эти эмбеддинги, получется много вариантов одного и того же объекта
2. Мультивекторные представления и Late Interaction. Более сложный механизм, предназначенный для моделей вроде ColBERT или ColPali, где один документ представляется не одним вектором, а набором векторов (например, по одному на каждый токен или фрагмент изображения). Если объединить эти эмбеддинги, получится представляемый ими объект.

MaxSim: Qdrant использует оператор MaxSim для вычисления сходства. Он находит наиболее похожий вектор в документе для каждого вектора в поисковом запросе и суммирует эти показатели

**ЗАЧЕМ:**

Можно же было бы создать для каждого ракурса свою точку в ВБД
1) Дедупликация в выдаче - не выдаст 3 одинаковые модели, потому что запрос похож на 3 ракурса одной модели. Всегда будут РАЗНЫЕ модели.
2) Управление данными (CRUD) - при необходимости обновить точку, надо будет обновить одну, а не искать все точки, принадлежащие этой модели
3) Производительность и индексы - используется специальный тип сжатия и индексации. В случае с max_sim Qdrant оптимизирует поиск так, чтобы не сравнивать запрос с каждым вектором в лоб, а использовать аппроксимацию.


In [68]:
lamoda_data_racurses = pd.read_csv(prefix_path+"lamoda_data.csv", sep=";")
print(len(lamoda_data_racurses))

1347


In [69]:
grouped = lamoda_data_racurses.groupby(['brand', 'model', 'category', 'color', 'description'])['title_photo'].apply(list).reset_index()
print(grouped)

                brand                                       model  \
0                ACBC  ACBC Кроссовки LOW TOP WOMAN ECO MATERIALS   
1               ASICS               ASICS Кроссовки GEL-KAYANO 32   
2               ASICS          ASICS Кроссовки GEL-KINETIC FLUENT   
3               Altra                            Altra Кроссовки    
4               Altra                  Altra Кроссовки Paradigm 7   
..                ...                                         ...   
139  adidas Originals          adidas Originals Кроссовки TOKYO W   
140        adidas Y-3         adidas Y-3 Кроссовки Y-3 A3 CONTROL   
141        adidas Y-3              adidas Y-3 Кроссовки Y-3 KAIWA   
142      adidas YEEZY            adidas YEEZY Кроссовки BOOST 700   
143      adidas YEEZY   adidas YEEZY Кроссовки YEEZY BOOST 350 V2   

             category  color  \
0    Низкие кроссовки  40124   
1    Низкие кроссовки  46422   
2    Низкие кроссовки  47051   
3    Низкие кроссовки  42084   
4    Низкие

In [95]:
def create_multivector_db(
        client: QdrantClient,
        collection_name: str, 
        emb_dim: int,
        data):
    
    unprocessable = 0
    
    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config={
                "photos": models.VectorParams(
                    size=emb_dim, 
                    distance=models.Distance.COSINE,
                    multivector_config=models.MultiVectorConfig(
                        comparator=models.MultiVectorComparator.MAX_SIM #позволяет хранить список векторов под одним именем
                    ) 
                )
            }
        )

        for idx, row in data.iterrows():
            try:
                multivector = []
                for racurs in row["title_photo"]:
                    try:
                        img_path = prefix_path + racurs
                        img_emb = get_image_embedding(img_path)
                        multivector.append(img_emb.tolist())
                    except:
                        continue

                point = models.PointStruct(
                    id=idx, 
                    vector={"photos": multivector}, 
                    payload={
                        "brand": row["brand"],
                        "model": row["model"],
                        "color": row["color"],
                        "path_to_photo": os.path.join(prefix_path, row["title_photo"][0]) # cохраняем путь для отображения!
                    }
                )

                client.upsert(
                    collection_name=collection_name, 
                    points=[point]
                    )
                    
            except: 
                unprocessable+=1
                continue
        print("unprocessable: ", unprocessable)
    else:
        print("Коллекция уже существует")

In [96]:
def get_image_embedding(img_path):
    image = Image.open(img_path).convert('RGB')

    preproc_image = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        image_emb = model.encode_image(preproc_image)
        
    # Нормализуем эмбеддинг (это важно для поиска в Qdrant)
    image_emb /= image_emb.norm(dim=-1, keepdim=True)
    
    return image_emb.cpu().numpy().flatten()

In [97]:
collection_name = "Sneakers_CLIP_multivector"

create_multivector_db(
    client,
    collection_name, 
    emb_dim = 512,
    data = grouped)

unprocessable:  0


In [100]:
def count_metrics(collection_name):

    recall_5 = 0
    recall_10 = 0
    accuracy = 0
    query_time = []
    N = 0
    unprocessable = []

    for image_path, reference_model in image_reference.items():
        try:
            start = time.perf_counter()
            query = get_image_embedding(prefix_path+image_path)
            candidates = client.query_points(
                                        collection_name=collection_name,
                                        query=query, 
                                        using = "photos",
                                        limit=10,           
                                        with_payload=True   # Возвращаем бренд, модель и путь из CSV
                                    ).points
            end = time.perf_counter()
        
            results = [candidate.payload.get("model") for candidate in candidates]
            if reference_model in results:
                recall_10 += 1
            if reference_model in results[:5]:
                recall_5 += 1
            if reference_model == results[0]:
                accuracy += 1
            query_time.append(end-start)
            N +=1

        except:
            unprocessable.append(image_path)
            continue

    print("Unprocessable: ", len(unprocessable))

    return recall_10 *100 / N, recall_5 * 100 / N, accuracy * 100 / N, np.mean(query_time)

In [104]:
recall_10_clip_multivectors, recall_5_clip_multivectors, accuracy_clip_multivectors, avg_query_time_clip_multivectors = count_metrics(collection_name)

df_data = {
    "Accuracy" : [accuracy_clip, accuracy_dino, accuracy_clip_multivectors],
    "Recall@5": [recall_5_clip, recall_5_dino, recall_5_clip_multivectors], 
    "Recall@10": [recall_10_clip, recall_10_dino, recall_10_clip_multivectors],
    "Avg_query_time": [avg_query_time_clip, avg_query_time_dino, avg_query_time_clip_multivectors]
}
  
table = pd.DataFrame(data=df_data)
table

Unprocessable:  0


,Accuracy,Recall@5,Recall@10,Avg_query_time
0,25.0,45.0,54.0,0.263479
1,3.0,14.0,26.0,0.195054
2,19.0,30.0,42.0,0.211860


In [105]:
def get_image_embedding(img_path):
    image = Image.open(img_path).convert('RGB')
    
    preproc_image = processor(images=image, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = model_dino(**preproc_image)
        image_emb = outputs.last_hidden_state[:, 0, :] # CLS-token
        # Нормализуем эмбеддинг (это важно для поиска в Qdrant)
        image_emb /= image_emb.norm(dim=-1, keepdim=True)
    
    return image_emb.cpu().numpy().flatten()

In [106]:
collection_name = "Sneakers_DINO_multivector"

create_multivector_db(
    client,
    collection_name, 
    emb_dim = 1024,
    data = grouped)

unprocessable:  0


In [107]:
recall_10_dino_multivectors, recall_5_dino_multivectors, accuracy_dino_multivectors, avg_query_time_dino_multivectors = count_metrics(collection_name)

df_data = {
    "Accuracy" : [accuracy_clip, accuracy_dino, accuracy_clip_multivectors, accuracy_dino_multivectors],
    "Recall@5": [recall_5_clip, recall_5_dino, recall_5_clip_multivectors, recall_5_dino_multivectors], 
    "Recall@10": [recall_10_clip, recall_10_dino, recall_10_clip_multivectors, recall_10_dino_multivectors],
    "Avg_query_time": [avg_query_time_clip, avg_query_time_dino, avg_query_time_clip_multivectors, avg_query_time_dino_multivectors]
}
  
table = pd.DataFrame(data=df_data, index=["CLIP-ViT", "DINOv2", "CLIP_multivector", "DINO_multivector"])
table

Unprocessable:  0


,Accuracy,Recall@5,Recall@10,Avg_query_time
CLIP-ViT,25.0,45.0,54.0,0.263479
DINOv2,3.0,14.0,26.0,0.195054
CLIP_multivector,19.0,30.0,42.0,0.211860
DINO_multivector,14.0,18.0,20.0,0.229882


## Preprocessing (crop)

## FT